# Recipe Chat Bot
### Start installing needed packages
### mounting your drive if needed


In [83]:
!pip install -q langchain langchain-community langchain-chroma
!pip install -q langchain_openai langchain-classic
!pip install -q langchain-huggingface huggingface_hub
!pip install -q langchain-text-splitters pypdf pymupdf chromadb

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load PDFs from Google Drive using PyMuPDF

In [55]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
#from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

PDF_FOLDER = "/content/drive/MyDrive/Recipe_RAG/recipe_pdfs"
DB_FOLDER = "/content/drive/MyDrive/Recipe_RAG/chroma_recipes_db"

# 1. Load PDFs from Google Drive using PyMuPDF
# We pass `silent_errors=True` to DirectoryLoader to avoid crash on bad files
loader = PyPDFDirectoryLoader(PDF_FOLDER, glob="**/*.pdf",
                              silent_errors=True)
#loader = DirectoryLoader(
 #   PDF_FOLDER,
 #   glob="**/*.pdf",
  #  loader_cls=PyMuPDFLoader,
  #  show_progress=True,
 #   silent_errors=True
#)
try:
  docs = loader.load()
  print(f"Successfully loaded {len(docs)} pages/documents!")
except Exception as e:
    print(f"An error occurred during loading: {e}")

Successfully loaded 845 pages/documents!


# Split into recipe-sized chunks


In [56]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Split into recipe-sized chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1800,
    chunk_overlap=250,
    separators=["\n\n", "\n", ".", " "]
)
chunks = text_splitter.split_documents(docs)
print(f"{len(chunks)} chunks created")

1573 chunks created


#### Print sample chunks to check how they split

In [58]:
# Inspect Chunk 10 and Chunk 11 (skipping early table-of-contents pages)

for i in range(10, 12):
    print(f"================ CHUNK {i} ================")
    print(f"Length: {len(chunks[i].page_content)} characters")
    print(f"Metadata: {chunks[i].metadata}")
    print("-" * 40)
    print(chunks[i].page_content)
    print("\n")

================ CHUNK 10 ================
Length: 72 characters
Metadata: {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2012-09-13T17:06:20-04:00', 'moddate': '2012-09-13T17:10:17-04:00', 'trapped': '/False', 'source': '/content/drive/MyDrive/Recipe_RAG/recipe_pdfs/MyPlate_RecipeBook2012_FINAL_SinglePages-5B8E06286B084B78B2EA5174E1711C5F.pdf', 'total_pages': 64, 'page': 9, 'page_label': '10'}
----------------------------------------
Preparation Time: 10 minutes
clementine, raisin & Goat cheese Sandwich
7


================ CHUNK 11 ================
Length: 1414 characters
Metadata: {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2012-09-13T17:06:20-04:00', 'moddate': '2012-09-13T17:10:17-04:00', 'trapped': '/False', 'source': '/content/drive/MyDrive/Recipe_RAG/recipe_pdfs/MyPlate_RecipeBook2012_FINAL_SinglePages-5B8E06286B084B78B2EA5174E1711C5F.pdf', 'total_pages': 64, 'page

# Create embeddings and store in ChromaDB
### persist it on the drive

In [70]:
import os
import torch
from google.colab import userdata

# Store your API key safely in Colab's left sidebar (Key icon -> Secrets)
os.environ["HUGGINGFACE_API_KEY"] = userdata.get('HUGGINGFACE_API_KEY')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print (f"device we will use = {device}")


device we will use = cpu


In [63]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Pass the detected device into HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

# uncomment following code if need to regenerate the vectors
# remove the old saved from drive else double chunks will be created
"""
# 3. Create and save Chroma DB permanently to Google Drive
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_FOLDER
)
print(f"Saved {len(chunks)} recipe chunks to Google Drive!")
"""
# 3. Load saved Chroma DB from Google Drive

vectorstore = Chroma(
    persist_directory=DB_FOLDER,
    embedding_function=embeddings
)

# 3. Verify that all 1,573 chunks loaded properly
print(f"Loaded {vectorstore._collection.count()} chunks from Google Drive!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved 1573 recipe chunks to Google Drive!


# Build RAG chain for QA with the bot

In [86]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
import os


# Build RAG chain pointing to HF serverless Inference API endpoint
llm = ChatOpenAI(
    model="Qwen/Qwen3.8-27B",
    api_key=userdata.get('HUGGINGFACE_API_KEY'),
    base_url="https://router.huggingface.co/v1",
    temperature=0
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True
)

In [91]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# 2. Initialize the native HuggingFaceEndpoint binding
endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3.5-9B",  # Or "meta-llama/Llama-3.1-8B-Instruct" / "openai/gpt-oss-120b"
    task="text-generation",
    max_new_tokens=512,
    temperature=0.1,
    provider="auto",             # Automatically selects an active free Inference Provider
    huggingfacehub_api_token=userdata.get("HUGGINGFACE_API_KEY")
)

# 3. Wrap in ChatHuggingFace so instruct/chat templates apply automatically
llm = ChatHuggingFace(llm=endpoint)

# 4. Connect to your RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True
)

### Query the recipe bot with a sample question

In [ ]:
# 5. Query!
result = qa_chain.invoke(
    "How do you make a Clementine, Raisin & Goat Cheese Sandwich?"
)
print("🤖 Bot Response:\n")
print(result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"  Source: {doc.metadata.get('source', 'Unknown')}, p.{doc.metadata.get('page', 'Unknown')}")

In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# 3. Load the free local Hugging Face model on T4 GPU
local_pipeline = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-3B-Instruct",
    task="text-generation",
    device_map="auto",
    model_kwargs={
        "torch_dtype": torch.float16 if device == "cuda" else torch.float32,
    },
    pipeline_kwargs={
        "max_new_tokens": 512,
        "temperature": 0.1,
        "do_sample": True,
        "return_full_text": False,
    },
)

llm = ChatHuggingFace(llm=local_pipeline)

# 4. Define the custom cooking PromptTemplate (must use {context} and {question})
recipe_template = """You are an expert chef and helpful recipe assistant.
Answer the user's question using ONLY the recipe context provided below.
Do not invent ingredients or steps that are not in the context.
If the requested recipe is not in the context, reply: "I couldn't find that recipe in your uploaded cookbooks."

When providing a recipe, format your response clearly as:
- **Recipe Name**
- **Prep / Cook Time** (if mentioned)
- **Ingredients** (bulleted list)
- **Instructions** (numbered steps)

Recipe Context:
{context}

User Question: {question}
RecipeBot's Answer:"""

QA_PROMPT = PromptTemplate(
    template=recipe_template,
    input_variables=["context", "question"]
)

# 5. Connect the custom prompt into RetrievalQA via chain_type_kwargs
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_PROMPT}
)

# 6. Test your recipe bot
result = qa_chain.invoke({"query": "Give me a recipe from the loaded PDFs."})
print(result["result"])
print("\nSources used:")
for doc in result["source_documents"]:
    print(f"- File: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')}")
